In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import sunpy.map
from skimage.transform import resize
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# --- Helper Functions (unchanged) ---


def load_fits_data(channel_dir: str) -> tuple[np.ndarray, dict]:
    """Load FITS data and metadata."""
    fits_files = [f for f in os.listdir(channel_dir) if f.endswith(".fits")]
    if not fits_files:
        raise FileNotFoundError(f"No FITS files found in directory: {channel_dir}")

    fits_path = os.path.join(channel_dir, fits_files[0])
    aia_map = sunpy.map.Map(fits_path)
    return aia_map.data, aia_map.meta


def create_circular_mask(data, metadata):
    """Creates a circular mask for the solar disk."""
    ny, nx = data.shape
    x_center = nx // 2
    y_center = ny // 2

    cdelt1 = metadata.get("cdelt1", 1.0)  # Arcsec/píxel en X
    solar_radius_arcsec = metadata.get("rsun_obs", 960.0)  # Radio solar en arcsec
    solar_radius_pixels = int(solar_radius_arcsec / abs(cdelt1))

    print(f"Metadata for channel: CDELT1={cdelt1}, RSUN_OBS={solar_radius_arcsec}")
    print(f"Solar radius in pixels: {solar_radius_pixels}")

    y, x = np.ogrid[:ny, :nx]
    distance_from_center = np.sqrt((x - x_center) ** 2 + (y - y_center) ** 2)

    mask = distance_from_center <= solar_radius_pixels
    return mask


def preprocess_image(data: np.ndarray, mask: np.ndarray, size: int = 512) -> np.ndarray:
    """Resize and mask the image."""
    resized_data = resize(data, (size, size), mode="reflect", anti_aliasing=True)
    resized_mask = resize(mask, (size, size), mode="reflect", anti_aliasing=False) > 0.5
    masked_data = resized_data.copy()
    masked_data[~resized_mask] = np.nan
    return masked_data


# --- Modified Functions ---


def prepare_data_concatenated(masked_data_list: list) -> np.ndarray:
    """
    Prepares the data for anomaly detection by concatenating channels and normalizing.

    Args:
        masked_data_list (list): List of masked image data (2D arrays) for each channel.

    Returns:
        np.ndarray: Data reshaped to (512*512, num_channels) and normalized.
    """

    # Stack the masked data along a new axis (channels)
    stacked_data = np.stack(masked_data_list, axis=-1)  # Shape: (512, 512, num_channels)

    # Reshape to (512*512, num_channels)
    reshaped_data = stacked_data.reshape((-1, len(masked_data_list)))

    # Robust scaling
    scaler = RobustScaler()  # initialize
    reshaped_data_scaled = scaler.fit_transform(reshaped_data)

    return reshaped_data_scaled


def detect_anomalies_isolation_forest_decision(data: np.ndarray, contamination: float = 0.05) -> np.ndarray:
    """
    Detects anomalies using Isolation Forest's decision_function.

    Args:
        data (np.ndarray):  Prepared data (samples x features).
        contamination (float): Expected proportion of anomalies.

    Returns:
        np.ndarray: Anomaly scores (higher = more normal, lower = more anomalous).
    """
    iso_forest = IsolationForest(contamination=contamination, random_state=42)
    iso_forest.fit(data)  # Fit the model
    anomaly_scores = iso_forest.decision_function(data)  # Get anomaly scores
    return anomaly_scores


def plot_comparison_decision(ax, masked_data_list, anomaly_scores, channel_names, threshold, i):
    """
    Plots the original image and overlays a mask highlighting anomalies based on the decision function.

    Args:
        ax (plt.Axes): Matplotlib Axes object for plotting.
        masked_data (np.ndarray): Original masked image data.
        anomaly_scores (np.ndarray): Anomaly scores from Isolation Forest's decision_function.
        channel (str): Name of the channel.
        threshold (float): Threshold to classify anomalies based on anomaly scores.
        i (int): Channel's index
    """
    masked_data = masked_data_list[i]
    channel = channel_names[i]
    # Reshape anomaly scores to the original image shape for visualization
    anomaly_map = anomaly_scores.reshape(masked_data.shape)

    # Create a mask for anomalies based on the threshold
    anomaly_mask = anomaly_map < threshold

    # Display the original image
    ax.imshow(
        masked_data,
        origin="lower",
        cmap="gray",
        vmin=np.nanpercentile(masked_data, 1),
        vmax=np.nanpercentile(masked_data, 99),
    )

    # Overlay the anomaly mask
    ax.imshow(anomaly_mask, origin="lower", cmap="Reds", alpha=0.5)
    ax.set_title(f"{channel}\nAnomaly Threshold: {threshold:.2f}", color="black")

ModuleNotFoundError: No module named 'skimage'

In [ ]:
# --- Main Execution ---

if __name__ == "__main__":
    data_dir = "./sdo_data/"  # Update this path if necessary
    channels = [
        d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))
    ]  # Correctly list channel directories
    num_channels = len(channels)

    # Parameters
    image_size = 512
    contamination = 0.05  # Adjust this value
    anomaly_threshold = -0.1  # Tune this threshold based on your data

    # 1. Load and Preprocess Images
    masked_data_list = []
    channel_names = []
    for channel_dir in channels:  # iterate over channels
        try:
            channel = channel_dir.split("_")[1]  # get channel name, example aia_171 -> 171

            channel_path = os.path.join(data_dir, channel_dir)
            data, metadata = load_fits_data(channel_path)
            mask = create_circular_mask(data, metadata)
            masked_data = preprocess_image(data, mask, image_size)
            masked_data_list.append(masked_data)
            channel_names.append(channel)

        except Exception as e:
            print(f"Error processing {channel_dir}: {e}")

    # 2. Prepare data concatenated for Isolation Forest
    prepared_data = prepare_data_concatenated(masked_data_list)

    # 3. Detect Anomalies
    anomaly_scores = detect_anomalies_isolation_forest_decision(prepared_data, contamination)

    # 4. Visualize Anomalies
    num_rows = 3
    num_cols = 3
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15))
    axes = axes.flatten()

    for i in range(num_channels):
        plot_comparison_decision(axes[i], masked_data_list, anomaly_scores, channel_names, anomaly_threshold, i)

    plt.tight_layout()
    plt.show()